# 01b · Predicción multiespecie desde embeddings de Perch

Este notebook reutiliza la base Hoplite producida por
`01_pipeline_embeddings.ipynb`. **No vuelve a leer ni a procesar los audios.**
Carga únicamente la cabeza clasificadora del mismo modelo Perch y la aplica a
los embeddings ya calculados.

Incluye dos enfoques complementarios:

1. **Búsqueda rápida:** consulta el índice ANN con el vector de cada especie,
   recalcula exactamente el score de los candidatos y conserva sólo los mejores.
   Es ideal para exploración, pero no garantiza cobertura exhaustiva por fecha o
   deployment.
2. **Inferencia exhaustiva:** multiplica cada embedding por la cabeza original de
   Perch para las especies seleccionadas. Es el resultado más parecido a ejecutar
   el modelo multiespecie sobre los audios, sin recalcular embeddings.

Ambos enfoques aplican una lista geográfica, guardan `logit` y
`score = sigmoid(logit)`, permiten un score mínimo y escriben resultados por
deployment y día u hora.

## Estructura de salida

Todas las rutas de salida son relativas al proyecto:

```text
outputs/multispecies/<run_name>/
├── run_metadata.json
├── species_filter/
│   ├── species_by_deployment.csv
│   ├── species_matched.csv
│   └── species_unmatched.csv
├── fast_search/deployment=<...>/date=<...>/predictions.parquet
├── exhaustive_inference/deployment=<...>/date=<...>/predictions.parquet
└── summaries/
    ├── fast_search_daily.csv
    ├── exhaustive_inference_daily.csv
    └── ...
```

Con `PARTITION_TIME = "hour"`, la carpeta `date=...` se sustituye por
`hour=YYYY-MM-DDTHH`. Parquet conserva tipos y ocupa mucho menos que CSV; los
resúmenes sí se guardan como CSV.

In [ ]:
from __future__ import annotations

import hashlib
import gc
import json
import math
import re
import unicodedata
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from scipy.special import expit
from tqdm.auto import tqdm

from perch_hoplite.db import sqlite_usearch_impl
from perch_hoplite.taxonomy import namespace_db
from perch_hoplite.zoo import model_configs

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. Configuración

Ajuste primero `DB_PATH`. La ruta propuesta coincide con los valores del
notebook 01 adjunto. La base puede permanecer en el disco de trabajo externo;
**todas las salidas** se crean dentro del proyecto.

Para el filtro geográfico reproducible se recomienda un CSV versionado en el
proyecto. Puede contener una lista para todo el país, una localidad o listas
distintas por deployment. La sección 4 crea una plantilla si no existe.

In [ ]:
def find_project_root(start: Path) -> Path:
    """Encuentra el primer padre con pyproject.toml."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(
        "No se encontró pyproject.toml. Ejecute el notebook dentro del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

# Base generada por 01_pipeline_embeddings.ipynb.
DB_PATH = Path(r"D:\hoplite_databases\C8\perch_embed_amistosaC8_ad")
MODEL_CHOICE = "perch_8"  # Debe ser exactamente el usado para crear la base.

# Identidad del análisis. Cambie el nombre al cambiar parámetros importantes.
RUN_NAME = "perch8_geographic_v1"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "multispecies" / RUN_NAME

# Filtro: "csv", "none", "geofence_country" o "geofence_coordinates".
# En perch-hoplite==1.0.2 use "csv" (recomendado) o "none".
GEOGRAPHIC_FILTER_MODE = "csv"
SPECIES_SCOPE_CSV = PROJECT_ROOT / "config" / "geographic_species.csv"
DEPLOYMENT_COORDINATES_CSV = PROJECT_ROOT / "config" / "deployment_coordinates.csv"
SCOPE_COUNTRY: str | None = "Costa Rica"
SCOPE_LOCALITY: str | None = None

# El manifiesto del notebook 01 permite recuperar deployment y fecha aun cuando
# Hoplite no tenga datetime completo. Use None para buscarlo junto a la base.
SAMPLING_MANIFEST: Path | None = None
MANIFEST_DATE_FORMAT = "%Y%m%d"

# Regex opcional para extraer fecha y hora del nombre del audio. Debe contener
# un grupo llamado "datetime". Ponga None si los Recording ya tienen datetime.
FILENAME_DATETIME_REGEX: str | None = r"(?P<datetime>\d{8}[_-]\d{6})"
FILENAME_DATETIME_FORMAT = "%Y%m%d_%H%M%S"

# Ejecución y particionado.
RUN_FAST_SEARCH = True
RUN_EXHAUSTIVE_INFERENCE = True
PARTITION_TIME = "day"  # "day" o "hour"
RESUME = True

# Enfoque 1: búsqueda aproximada + reranking exacto.
FAST_RETRIEVAL_PER_SPECIES = 2_000
FAST_MIN_SCORE = 0.05
FAST_MAX_PER_SPECIES_PER_PARTITION = 50

# Enfoque 2: clasificación de todas las ventanas.
EXHAUSTIVE_MIN_SCORE = 0.05
MAX_PREDICTIONS_PER_WINDOW: int | None = 20
EMBEDDING_BATCH_SIZE = 512
WINDOW_METADATA_CHUNK_SIZE = 50_000

# Ajustes comunes.
SPECIES_ALWAYS_INCLUDE: tuple[str, ...] = ()  # perch_label o nombre científico
SPECIES_EXCLUDE: tuple[str, ...] = ()
OVERWRITE_PARTITIONS = False

print("Proyecto:", PROJECT_ROOT)
print("Base Hoplite:", DB_PATH)
print("Salidas:", OUTPUT_ROOT)

In [ ]:
@dataclass(frozen=True)
class RunSettings:
    db_path: str
    model_choice: str
    geographic_filter_mode: str
    scope_country: str | None
    scope_locality: str | None
    partition_time: str
    fast_retrieval_per_species: int
    fast_min_score: float
    fast_max_per_species_per_partition: int
    exhaustive_min_score: float
    max_predictions_per_window: int | None


def validate_settings() -> None:
    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"No existe DB_PATH: {DB_PATH}\n"
            "Copie la ruta exacta impresa por 01_pipeline_embeddings.ipynb."
        )
    if PARTITION_TIME not in {"day", "hour"}:
        raise ValueError("PARTITION_TIME debe ser 'day' o 'hour'.")
    if GEOGRAPHIC_FILTER_MODE not in {
        "csv", "none", "geofence_country", "geofence_coordinates"
    }:
        raise ValueError("GEOGRAPHIC_FILTER_MODE no reconocido.")
    for name, value in {
        "FAST_MIN_SCORE": FAST_MIN_SCORE,
        "EXHAUSTIVE_MIN_SCORE": EXHAUSTIVE_MIN_SCORE,
    }.items():
        if not 0.0 <= value <= 1.0:
            raise ValueError(f"{name} debe estar entre 0 y 1.")
    if EMBEDDING_BATCH_SIZE < 1:
        raise ValueError("EMBEDDING_BATCH_SIZE debe ser positivo.")


validate_settings()
for folder in (
    OUTPUT_ROOT,
    OUTPUT_ROOT / "species_filter",
    OUTPUT_ROOT / "summaries",
):
    folder.mkdir(parents=True, exist_ok=True)

## 2. Abrir Hoplite y construir la tabla de ventanas

La base se abre en modo de sólo lectura. Se unen las tablas de Hoplite con
`sampling_audit/sample_manifest.csv` del notebook 01. Para particionar por hora
se necesita una hora real en `recording.datetime` o extraíble del nombre; el
notebook no inventa `00:00` cuando sólo conoce el día.

In [ ]:
print("Abriendo la base Hoplite en modo de solo lectura...")
db = sqlite_usearch_impl.SQLiteUSearchDB.create(str(DB_PATH), readonly=True)
print("Base Hoplite abierta.")

deployments_df = pd.read_sql_query(
    """
    SELECT id AS deployment_id, name AS deployment, project, latitude, longitude
    FROM deployments
    """,
    db.db,
)
recordings_df = pd.read_sql_query(
    """
    SELECT id AS recording_id, filename, datetime AS recording_datetime,
           deployment_id
    FROM recordings
    """,
    db.db,
)
print(
    f"Metadatos SQL: {len(deployments_df):,} deployments y "
    f"{len(recordings_df):,} grabaciones."
)

def normalize_path(value: object) -> str:
    return str(value).strip().replace("\\", "/").casefold()


manifest_path = (
    SAMPLING_MANIFEST
    if SAMPLING_MANIFEST is not None
    else DB_PATH / "sampling_audit" / "sample_manifest.csv"
)

# Toda la limpieza de rutas y fechas se hace primero sobre recordings_df. Una
# grabación puede tener decenas o cientos de ventanas; hacerlo después del merge
# repetiría regex, conversiones y strings para cada ventana y puede agotar RAM.
recordings_df = recordings_df.merge(
    deployments_df,
    on="deployment_id",
    how="left",
    validate="many_to_one",
)

if manifest_path.exists():
    manifest_df = pd.read_csv(
        manifest_path,
        dtype={"relative_path": "string", "deployment": "string", "date": "string"},
    )
    if "relative_path" not in manifest_df.columns:
        raise ValueError(f"Falta relative_path en {manifest_path}")
    manifest_df["_path_key"] = manifest_df["relative_path"].map(normalize_path)
    recordings_df["_path_key"] = recordings_df["filename"].map(normalize_path)
    keep_manifest = [
        column for column in
        ("_path_key", "relative_path", "deployment", "date", "year_month")
        if column in manifest_df.columns
    ]
    manifest_unique = manifest_df[keep_manifest].drop_duplicates("_path_key")
    recordings_df = recordings_df.merge(
        manifest_unique,
        on="_path_key",
        how="left",
        suffixes=("", "_manifest"),
        validate="many_to_one",
    )
    if "deployment_manifest" in recordings_df:
        recordings_df["deployment"] = recordings_df["deployment"].fillna(
            recordings_df["deployment_manifest"]
        )
    # Fallback seguro cuando Hoplite guardó una ruta absoluta: sólo usa nombres
    # de archivo que sean únicos en todo el manifiesto.
    recordings_df["_basename_key"] = recordings_df["filename"].map(
        lambda value: Path(str(value)).name.casefold()
    )
    manifest_df["_basename_key"] = manifest_df["relative_path"].map(
        lambda value: Path(str(value)).name.casefold()
    )
    unique_basenames = manifest_df.loc[
        ~manifest_df["_basename_key"].duplicated(keep=False)
    ].set_index("_basename_key")
    for column in ("relative_path", "date", "year_month"):
        if column in unique_basenames and column in recordings_df:
            recordings_df[column] = recordings_df[column].fillna(
                recordings_df["_basename_key"].map(unique_basenames[column])
            )
else:
    manifest_df = pd.DataFrame()
    recordings_df["relative_path"] = recordings_df["filename"]
    print("Aviso: no se encontró el manifiesto de muestreo:", manifest_path)


recordings_df["recording_datetime"] = pd.to_datetime(
    recordings_df["recording_datetime"], errors="coerce"
)
if FILENAME_DATETIME_REGEX:
    filename_only = recordings_df["filename"].astype("string").str.replace(
        r"^.*[\\/]", "", regex=True
    )
    extracted_datetime = filename_only.str.extract(
        FILENAME_DATETIME_REGEX, expand=False
    )
    if isinstance(extracted_datetime, pd.DataFrame):
        extracted_datetime = extracted_datetime["datetime"]
    extracted_datetime = extracted_datetime.astype("string").str.replace(
        "-", "_", regex=False
    )
    parsed_from_filename = pd.to_datetime(
        extracted_datetime,
        format=FILENAME_DATETIME_FORMAT,
        errors="coerce",
    )
    recordings_df["recording_datetime"] = recordings_df[
        "recording_datetime"
    ].fillna(parsed_from_filename)
    del filename_only, extracted_datetime, parsed_from_filename

if "date" in recordings_df:
    raw_manifest_date = recordings_df["date"].astype("string")
    recordings_df["manifest_date"] = pd.to_datetime(
        raw_manifest_date, format=MANIFEST_DATE_FORMAT, errors="coerce"
    )
    needs_fallback = recordings_df["manifest_date"].isna() & raw_manifest_date.notna()
    if needs_fallback.any():
        recordings_df.loc[needs_fallback, "manifest_date"] = pd.to_datetime(
            raw_manifest_date.loc[needs_fallback], errors="coerce"
        )
    del raw_manifest_date, needs_fallback
else:
    recordings_df["manifest_date"] = pd.NaT

recordings_df["relative_path"] = recordings_df.get(
    "relative_path", recordings_df["filename"]
).fillna(recordings_df["filename"])
recordings_df["deployment"] = recordings_df["deployment"].fillna(
    "unknown_deployment"
)
recordings_df = recordings_df.drop(
    columns=[
        "_path_key", "_basename_key", "deployment_manifest", "date", "year_month",
        "latitude", "longitude",
    ],
    errors="ignore",
)

print(f"Grabaciones en metadatos: {len(recordings_df):,}")

### 2.1 Leer ventanas sin crear objetos `Window`

`db.get_all_windows()` crea un objeto Python por ventana y después una segunda
copia al construir el DataFrame. En bases grandes ese pico puede matar el
kernel. Esta lectura consulta SQLite por bloques y escribe directamente en
cuatro arreglos NumPy preasignados.

In [ ]:
count_cursor = db.db.cursor()
n_windows = int(count_cursor.execute("SELECT COUNT(*) FROM windows").fetchone()[0])
count_cursor.close()
if n_windows == 0:
    raise RuntimeError("La base no contiene ventanas de embeddings.")

window_id_array = np.empty(n_windows, dtype=np.int64)
recording_id_array = np.empty(n_windows, dtype=np.int64)
window_start_array = np.empty(n_windows, dtype=np.float32)
window_end_array = np.empty(n_windows, dtype=np.float32)

window_cursor = db.db.cursor()
window_cursor.execute(
    """
    SELECT id, recording_id,
           GET_OFFSET_START(offsets), GET_OFFSET_END(offsets)
    FROM windows
    ORDER BY id
    """
)
position = 0
progress = tqdm(total=n_windows, desc="Metadatos de ventanas", unit="ventana")
try:
    while True:
        rows = window_cursor.fetchmany(WINDOW_METADATA_CHUNK_SIZE)
        if not rows:
            break
        end = position + len(rows)
        window_id_array[position:end] = np.fromiter(
            (row[0] for row in rows), dtype=np.int64, count=len(rows)
        )
        recording_id_array[position:end] = np.fromiter(
            (row[1] for row in rows), dtype=np.int64, count=len(rows)
        )
        window_start_array[position:end] = np.fromiter(
            (row[2] for row in rows), dtype=np.float32, count=len(rows)
        )
        window_end_array[position:end] = np.fromiter(
            (row[3] for row in rows), dtype=np.float32, count=len(rows)
        )
        position = end
        progress.update(len(rows))
        del rows
finally:
    progress.close()
    window_cursor.close()

if position != n_windows:
    raise RuntimeError(
        f"SQLite reportó {n_windows:,} ventanas, pero se leyeron {position:,}."
    )

windows_df = pd.DataFrame(
    {
        "window_id": window_id_array,
        "recording_id": recording_id_array,
        "window_start_s": window_start_array,
        "window_end_s": window_end_array,
    },
    copy=False,
)
del window_id_array, recording_id_array, window_start_array, window_end_array
gc.collect()
print(
    f"Ventanas numéricas leídas: {len(windows_df):,}; memoria: "
    f"{windows_df.memory_usage(index=True, deep=True).sum() / 1024**2:,.1f} MiB"
)

### 2.2 Unir metadatos y derivar fecha/hora

In [ ]:
recording_columns = [
    "recording_id", "filename", "recording_datetime", "deployment", "project",
    "relative_path", "manifest_date",
]
windows_df = windows_df.merge(
    recordings_df[recording_columns],
    on="recording_id",
    how="left",
    validate="many_to_one",
    sort=False,
)

windows_df["window_datetime"] = windows_df["recording_datetime"] + pd.to_timedelta(
    windows_df["window_start_s"], unit="s"
)
window_date = windows_df["window_datetime"].dt.normalize().fillna(
    windows_df["manifest_date"]
)
windows_df["date"] = window_date.dt.strftime("%Y-%m-%d").astype("category")
windows_df["hour"] = (
    windows_df["window_datetime"].dt.strftime("%Y-%m-%dT%H").astype("category")
)
windows_df = windows_df.drop(
    columns=["recording_datetime", "manifest_date"], errors="ignore"
)
for column in ("deployment", "project", "filename", "relative_path"):
    windows_df[column] = windows_df[column].astype("category")
del window_date
gc.collect()

if PARTITION_TIME == "hour" and windows_df["hour"].isna().any():
    missing = int(windows_df["hour"].isna().sum())
    raise ValueError(
        f"Hay {missing:,} ventanas sin hora. Corrija FILENAME_DATETIME_REGEX, "
        "complete Recording.datetime o use PARTITION_TIME='day'."
    )
if windows_df["date"].isna().any():
    missing = int(windows_df["date"].isna().sum())
    raise ValueError(
        f"Hay {missing:,} ventanas sin fecha. Revise el manifiesto o el regex."
    )

windows_df.set_index("window_id", drop=False, inplace=True, verify_integrity=True)

print(f"Deployments: {windows_df['deployment'].nunique():,}")
print(f"Grabaciones: {windows_df['recording_id'].nunique():,}")
print(f"Ventanas: {len(windows_df):,}")
print(f"Fechas: {windows_df['date'].min()} → {windows_df['date'].max()}")
print(
    "Memoria de windows_df:",
    f"{windows_df.memory_usage(index=True, deep=True).sum() / 1024**2:,.1f} MiB",
)
display(
    windows_df[
        [
            "window_id", "deployment", "relative_path", "window_start_s",
            "window_end_s", "window_datetime", "date", "hour",
        ]
    ].head()
)

## 3. Cargar la cabeza original de Perch

`get_classifier_head` extrae de los weights originales una matriz `W` y un
bias `b`. Para cada embedding `e`, el notebook calcula exactamente:

$$\text{logit}=eW+b, \qquad \text{score}=\sigma(\text{logit})$$

El score es útil para umbrales y ordenamiento, pero no debe interpretarse como
una probabilidad ecológica calibrada. La equivalencia con la inferencia desde
audio requiere que `MODEL_CHOICE`, la versión del modelo y los embeddings de la
base sean los mismos.

In [ ]:
model = model_configs.load_model_by_name(MODEL_CHOICE)
if "label" not in model.class_list:
    raise KeyError(f"El modelo no contiene una lista 'label': {model.class_list.keys()}")

model_class_list = model.class_list["label"]
model_namespace = model_class_list.namespace
model_labels = tuple(model_class_list.classes)
label_order = {label: index for index, label in enumerate(model_labels)}

taxonomy = namespace_db.load_db()


def build_label_to_scientific(
    source_namespace: str, labels: Sequence[str]
) -> dict[str, str]:
    """Convierte labels del modelo a binomios usando los mappings de Hoplite."""
    # Namespaces que ya contienen nombres científicos.
    scientific_namespace_tokens = ("clements", "xenocanto", "ioc_", "inat")
    if any(token in source_namespace.casefold() for token in scientific_namespace_tokens):
        return {label: label for label in labels}

    # Caso eBird: label/subespecie -> código de especie -> nombre Clements.
    to_species_name = f"{source_namespace}_to_species"
    if to_species_name in taxonomy.mappings:
        to_species = taxonomy.mappings[to_species_name]
        species_namespace = to_species.target_namespace
        clements_candidates = [
            mapping
            for mapping in taxonomy.mappings.values()
            if mapping.target_namespace == species_namespace
            and "clements" in mapping.source_namespace.casefold()
        ]
        if clements_candidates:
            clements_to_species = clements_candidates[0].mapped_pairs
            species_to_scientific = {
                species_code: scientific_name
                for scientific_name, species_code in clements_to_species.items()
            }
            return {
                label: species_to_scientific[species_code]
                for label, species_code in to_species.mapped_pairs.items()
                if label in label_order and species_code in species_to_scientific
            }

    # Si el source ya es el namespace de códigos de especie, invierte Clements.
    clements_candidates = [
        mapping
        for mapping in taxonomy.mappings.values()
        if mapping.target_namespace == source_namespace
        and "clements" in mapping.source_namespace.casefold()
    ]
    if clements_candidates:
        return {
            species_code: scientific_name
            for scientific_name, species_code in clements_candidates[0].mapped_pairs.items()
            if species_code in label_order
        }

    print(
        f"Aviso: no se encontró un mapping científico para {source_namespace}; "
        "se tratarán las etiquetas del modelo como nombres científicos."
    )
    return {label: label for label in labels}


label_to_scientific = build_label_to_scientific(model_namespace, model_labels)

model_taxonomy_df = pd.DataFrame(
    {
        "perch_label": model_labels,
        "scientific_name": [label_to_scientific.get(label) for label in model_labels],
        "model_index": np.arange(len(model_labels), dtype=np.int32),
    }
)

print("Namespace:", model_namespace)
print("Clases de la cabeza label:", f"{len(model_labels):,}")
display(model_taxonomy_df.head())

## 4. Lista geográfica

El modo recomendado con el entorno fijado (`perch-hoplite==1.0.2`) es `csv`.
El archivo puede tener estas columnas:

| columna | uso |
|---|---|
| `scientific_name` | nombre científico; se convierte al namespace del modelo |
| `perch_label` | etiqueta exacta del modelo; tiene prioridad |
| `deployment` | opcional; vacío o `*` aplica a todos |
| `country` | opcional; permite elegir un país dentro de un catálogo mayor |
| `locality` | opcional; permite elegir una localidad |
| `include` | opcional; `false` excluye la fila |

El mismo CSV puede almacenar varias listas geográficas. Si no existe, la
siguiente celda crea sólo la plantilla y se detiene para que usted la complete.

Los modos `geofence_country` y `geofence_coordinates` se activan
automáticamente si una versión futura/experimental de Perch aporta
`perch_hoplite.geofence`; no son parte de 1.0.2.

In [ ]:
def normalized_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"\s+", " ", text.strip()).casefold()


def create_csv_template(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    template = pd.DataFrame(
        columns=[
            "scientific_name", "perch_label", "deployment", "country",
            "locality", "include", "source",
        ]
    )
    template.to_csv(path, index=False)


def geographic_rows_from_csv() -> pd.DataFrame:
    if not SPECIES_SCOPE_CSV.exists():
        create_csv_template(SPECIES_SCOPE_CSV)
        raise FileNotFoundError(
            f"Se creó la plantilla {SPECIES_SCOPE_CSV}. Complétela con una "
            "especie por fila y vuelva a ejecutar esta celda."
        )
    rows = pd.read_csv(SPECIES_SCOPE_CSV)
    if not {"scientific_name", "perch_label"}.intersection(rows.columns):
        raise ValueError("El CSV requiere scientific_name y/o perch_label.")
    for column in ("scientific_name", "perch_label", "deployment"):
        if column not in rows:
            rows[column] = pd.NA
    if "include" in rows:
        excluded_values = {"0", "false", "no", "n"}
        rows = rows[
            ~rows["include"].map(normalized_text).isin(excluded_values)
        ].copy()
    if SCOPE_COUNTRY and "country" in rows:
        wanted = normalized_text(SCOPE_COUNTRY)
        rows = rows[rows["country"].map(normalized_text).isin({"", wanted})]
    if SCOPE_LOCALITY and "locality" in rows:
        wanted = normalized_text(SCOPE_LOCALITY)
        rows = rows[rows["locality"].map(normalized_text).isin({"", wanted})]
    return rows.reset_index(drop=True)


def geographic_rows_from_optional_geofence() -> pd.DataFrame:
    try:
        from perch_hoplite.geofence import geofence_inference
    except ImportError as exc:
        raise RuntimeError(
            "La versión instalada de perch-hoplite no contiene geofence. "
            "Use GEOGRAPHIC_FILTER_MODE='csv' y guarde la lista geográfica "
            "dentro de config/geographic_species.csv."
        ) from exc

    geofence = geofence_inference.GeofenceInference()
    frames: list[pd.DataFrame] = []
    if GEOGRAPHIC_FILTER_MODE == "geofence_country":
        if not SCOPE_COUNTRY:
            raise ValueError("Defina SCOPE_COUNTRY.")
        result = geofence.get_species_for_country(SCOPE_COUNTRY)
        frame = pd.DataFrame(result).rename(columns={"species_id": "scientific_name"})
        frame["deployment"] = "*"
        frames.append(frame)
    else:
        if DEPLOYMENT_COORDINATES_CSV.exists():
            coords = pd.read_csv(DEPLOYMENT_COORDINATES_CSV)
        else:
            coords = deployments_df[["deployment", "latitude", "longitude"]]
        required = {"deployment", "latitude", "longitude"}
        if not required.issubset(coords.columns) or coords[list(required)].isna().any().any():
            raise ValueError(
                "Se requieren deployment, latitude y longitude completos en "
                f"{DEPLOYMENT_COORDINATES_CSV}."
            )
        for row in coords.itertuples(index=False):
            result = geofence.get_species_for_lat_lon(row.latitude, row.longitude)
            frame = pd.DataFrame(result).rename(
                columns={"species_id": "scientific_name"}
            )
            frame["deployment"] = row.deployment
            frames.append(frame)
    rows = pd.concat(frames, ignore_index=True)
    rows["perch_label"] = pd.NA
    return rows


def requested_geographic_rows() -> pd.DataFrame:
    if GEOGRAPHIC_FILTER_MODE == "none":
        return pd.DataFrame(
            {
                "perch_label": model_labels,
                "scientific_name": [label_to_scientific.get(x) for x in model_labels],
                "deployment": "*",
            }
        )
    if GEOGRAPHIC_FILTER_MODE == "csv":
        return geographic_rows_from_csv()
    return geographic_rows_from_optional_geofence()


requested_species_df = requested_geographic_rows()
if requested_species_df.empty:
    raise ValueError("La lista geográfica quedó vacía después de aplicar los filtros.")

In [ ]:
scientific_to_labels: dict[str, list[str]] = {}
for item in model_taxonomy_df.dropna(subset=["scientific_name"]).itertuples(index=False):
    scientific_to_labels.setdefault(normalized_text(item.scientific_name), []).append(
        item.perch_label
    )


def labels_for_request(perch_label: object, scientific_name: object) -> list[str]:
    exact = "" if pd.isna(perch_label) else str(perch_label).strip()
    if exact:
        return [exact] if exact in label_order else []
    return scientific_to_labels.get(normalized_text(scientific_name), [])


matched_rows: list[dict] = []
unmatched_rows: list[dict] = []
for request_index, row in requested_species_df.iterrows():
    raw_deployment = row.get("deployment", "*")
    deployment = (
        "*" if pd.isna(raw_deployment) or not str(raw_deployment).strip()
        else str(raw_deployment).strip()
    )
    matches = labels_for_request(row.get("perch_label"), row.get("scientific_name"))
    if not matches:
        unmatched_rows.append(
            {
                "request_index": request_index,
                "deployment": deployment,
                "requested_perch_label": row.get("perch_label"),
                "requested_scientific_name": row.get("scientific_name"),
            }
        )
        continue
    for label in matches:
        matched_rows.append(
            {
                "request_index": request_index,
                "deployment": deployment,
                "perch_label": label,
                "scientific_name": label_to_scientific.get(label),
            }
        )


def resolve_manual_labels(values: Sequence[str]) -> set[str]:
    resolved: set[str] = set()
    for value in values:
        if value in label_order:
            resolved.add(value)
        else:
            resolved.update(scientific_to_labels.get(normalized_text(value), []))
    return resolved


always_include = resolve_manual_labels(SPECIES_ALWAYS_INCLUDE)
always_exclude = resolve_manual_labels(SPECIES_EXCLUDE)
deployment_names = sorted(
    recordings_df["deployment"].dropna().astype(str).unique()
)

matched_df = pd.DataFrame(
    matched_rows,
    columns=["request_index", "deployment", "perch_label", "scientific_name"],
)
unmatched_df = pd.DataFrame(
    unmatched_rows,
    columns=[
        "request_index", "deployment", "requested_perch_label",
        "requested_scientific_name",
    ],
)
by_deployment: list[dict] = []
for deployment in deployment_names:
    if matched_df.empty:
        labels = set()
    else:
        labels = set(
            matched_df.loc[
                matched_df["deployment"].isin({"*", deployment}), "perch_label"
            ]
        )
    labels = (labels | always_include) - always_exclude
    for label in sorted(labels, key=label_order.get):
        by_deployment.append(
            {
                "deployment": deployment,
                "perch_label": label,
                "scientific_name": label_to_scientific.get(label),
            }
        )

species_by_deployment_df = pd.DataFrame(by_deployment)
missing_deployments = sorted(
    set(deployment_names) - set(species_by_deployment_df.get("deployment", []))
)
if missing_deployments:
    raise ValueError(
        "No hay especies aplicables para estos deployments: "
        + ", ".join(missing_deployments[:20])
    )

audit_dir = OUTPUT_ROOT / "species_filter"
matched_df.to_csv(audit_dir / "species_matched.csv", index=False)
unmatched_df.to_csv(audit_dir / "species_unmatched.csv", index=False)
species_by_deployment_df.to_csv(
    audit_dir / "species_by_deployment.csv", index=False
)

union_labels = sorted(
    species_by_deployment_df["perch_label"].unique(), key=label_order.get
)
print(f"Solicitudes geográficas: {len(requested_species_df):,}")
print(f"Etiquetas Perch seleccionadas: {len(union_labels):,}")
print(f"Solicitudes sin match: {len(unmatched_df):,}")
display(species_by_deployment_df.groupby("deployment").size().rename("n_species"))
if not unmatched_df.empty:
    display(unmatched_df.head(20))

## 5. Extraer weights, validar dimensiones y fijar la corrida

El archivo `run_metadata.json` contiene un hash de la configuración y la lista
efectiva de clases. `RESUME=True` sólo reutiliza particiones cuando ese hash es
idéntico, lo que evita mezclar resultados de umbrales o listas diferentes.

In [ ]:
found_labels, classifier_weights, classifier_biases = model.get_classifier_head(
    union_labels
)
found_labels = list(found_labels)
if found_labels != union_labels:
    missing = sorted(set(union_labels) - set(found_labels))
    raise RuntimeError(f"La cabeza no devolvió {len(missing)} etiquetas: {missing[:20]}")

classifier_weights = np.asarray(classifier_weights, dtype=np.float32)
classifier_biases = np.asarray(classifier_biases, dtype=np.float32)
head_index = {label: index for index, label in enumerate(found_labels)}

sample_embedding = np.asarray(
    db.get_embeddings_batch([int(windows_df.iloc[0]["window_id"])]),
    dtype=np.float32,
)
if sample_embedding.ndim != 2 or sample_embedding.shape[1] != classifier_weights.shape[0]:
    raise ValueError(
        "Dimensión incompatible entre la base y la cabeza: "
        f"embeddings={sample_embedding.shape}, weights={classifier_weights.shape}. "
        "Verifique MODEL_CHOICE."
    )

settings = RunSettings(
    db_path=str(DB_PATH.resolve()),
    model_choice=MODEL_CHOICE,
    geographic_filter_mode=GEOGRAPHIC_FILTER_MODE,
    scope_country=SCOPE_COUNTRY,
    scope_locality=SCOPE_LOCALITY,
    partition_time=PARTITION_TIME,
    fast_retrieval_per_species=FAST_RETRIEVAL_PER_SPECIES,
    fast_min_score=FAST_MIN_SCORE,
    fast_max_per_species_per_partition=FAST_MAX_PER_SPECIES_PER_PARTITION,
    exhaustive_min_score=EXHAUSTIVE_MIN_SCORE,
    max_predictions_per_window=MAX_PREDICTIONS_PER_WINDOW,
)
fingerprint_payload = {
    "settings": asdict(settings),
    "species_by_deployment": species_by_deployment_df.to_dict("records"),
}
config_hash = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True, ensure_ascii=False).encode()
).hexdigest()

metadata_path = OUTPUT_ROOT / "run_metadata.json"
if metadata_path.exists() and RESUME:
    existing = json.loads(metadata_path.read_text(encoding="utf-8"))
    if existing.get("config_hash") != config_hash:
        raise ValueError(
            "RUN_NAME ya contiene otra configuración. Cambie RUN_NAME o restaure "
            "los parámetros anteriores."
        )

run_metadata = {
    **fingerprint_payload,
    "config_hash": config_hash,
    "created_or_validated_utc": datetime.now(timezone.utc).isoformat(),
    "n_windows": int(len(windows_df)),
    "n_recordings": int(windows_df["recording_id"].nunique()),
    "n_selected_labels": len(found_labels),
    "embedding_dim": int(classifier_weights.shape[0]),
    "model_namespace": model_namespace,
    "score_definition": "sigmoid(embedding @ classifier_weights + classifier_bias)",
}


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


atomic_write_text(
    metadata_path,
    json.dumps(run_metadata, indent=2, ensure_ascii=False, sort_keys=True),
)
print("Config hash:", config_hash[:16])
print("Weights:", classifier_weights.shape, "Bias:", classifier_biases.shape)

## 6. Funciones de selección, metadatos y escritura

In [ ]:
PREDICTION_COLUMNS = [
    "window_id", "recording_id", "deployment", "project", "filename",
    "relative_path", "window_start_s", "window_end_s", "window_datetime",
    "date", "hour", "perch_label", "scientific_name", "logit", "score",
    "model_choice", "method", "config_hash",
]


def score_to_logit(score: float) -> float:
    if score <= 0:
        return -np.inf
    if score >= 1:
        return np.inf
    return math.log(score / (1.0 - score))


def safe_component(value: object) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^A-Za-z0-9._-]+", "-", text).strip("-._")
    return text or "unknown"


def partition_column() -> str:
    return "date" if PARTITION_TIME == "day" else "hour"


def partition_path(base: Path, deployment: object, period: object) -> Path:
    key = partition_column()
    return (
        base
        / f"deployment={safe_component(deployment)}"
        / f"{key}={safe_component(period)}"
        / "predictions.parquet"
    )


def empty_prediction_frame() -> pd.DataFrame:
    return pd.DataFrame({column: pd.Series(dtype="object") for column in PREDICTION_COLUMNS})


def atomic_write_parquet(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp.parquet")
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)


def select_from_logits(
    logits: np.ndarray,
    labels: Sequence[str],
    min_score: float,
    top_k: int | None,
) -> pd.DataFrame:
    """Devuelve índices de fila/clase que superan el umbral."""
    logits = np.asarray(logits, dtype=np.float32)
    n_rows, n_classes = logits.shape
    threshold = score_to_logit(min_score)
    if n_rows == 0 or n_classes == 0:
        return pd.DataFrame(columns=["_row", "perch_label", "logit"])

    if top_k is not None and top_k < n_classes:
        k = max(1, int(top_k))
        candidate_cols = np.argpartition(logits, n_classes - k, axis=1)[:, -k:]
        candidate_logits = np.take_along_axis(logits, candidate_cols, axis=1)
        order = np.argsort(-candidate_logits, axis=1)
        candidate_cols = np.take_along_axis(candidate_cols, order, axis=1)
        candidate_logits = np.take_along_axis(candidate_logits, order, axis=1)
        keep = candidate_logits >= threshold
        rows, positions = np.where(keep)
        cols = candidate_cols[rows, positions]
        values = candidate_logits[rows, positions]
    else:
        rows, cols = np.where(logits >= threshold)
        values = logits[rows, cols]

    return pd.DataFrame(
        {
            "_row": rows.astype(np.int64),
            "perch_label": np.asarray(labels, dtype=object)[cols],
            "logit": values.astype(np.float32),
        }
    )


metadata_columns = [
    "window_id", "recording_id", "deployment", "project", "filename",
    "relative_path", "window_start_s", "window_end_s", "window_datetime",
    "date", "hour",
]
def attach_metadata(
    selected: pd.DataFrame,
    batch_window_ids: Sequence[int],
    method: str,
) -> pd.DataFrame:
    if selected.empty:
        return empty_prediction_frame()
    ids = np.asarray(batch_window_ids, dtype=np.int64)[selected.pop("_row").to_numpy()]
    metadata = windows_df.loc[ids, metadata_columns].reset_index(drop=True)
    output = pd.concat([metadata, selected.reset_index(drop=True)], axis=1)
    output["scientific_name"] = output["perch_label"].map(label_to_scientific)
    output["score"] = expit(output["logit"].to_numpy(dtype=np.float32))
    output["model_choice"] = MODEL_CHOICE
    output["method"] = method
    output["config_hash"] = config_hash
    return output[PREDICTION_COLUMNS]


allowed_pairs = set(
    species_by_deployment_df[["deployment", "perch_label"]].itertuples(
        index=False, name=None
    )
)

## 7. Enfoque 1 — búsqueda rápida

Para cada especie se consulta ANN con su vector clasificador. El bias no cambia
el orden de vecinos; luego se recuperan esos embeddings, se recalculan
`embedding @ weight + bias` y se aplica el umbral exacto. Sólo se guarda el top
por especie y partición.

Esta ruta es muy eficiente para descubrir candidatos. Como la consulta ANN es
global, `FAST_RETRIEVAL_PER_SPECIES` debe crecer si la base contiene muchos
deployments o fechas; no garantiza candidatos para cada partición. Use el
enfoque exhaustivo para estimaciones de ausencia o cobertura completa.

In [ ]:
def run_fast_search() -> pd.DataFrame:
    base = OUTPUT_ROOT / "fast_search"
    candidates: list[pd.DataFrame] = []

    for class_index, label in enumerate(tqdm(found_labels, desc="ANN por especie")):
        results = db.search(
            classifier_weights[:, class_index],
            search_list_size=FAST_RETRIEVAL_PER_SPECIES,
            approximate=True,
        )
        window_ids = [int(result.window_id) for result in results]
        if not window_ids:
            continue
        embeddings = np.asarray(db.get_embeddings_batch(window_ids), dtype=np.float32)
        logits = embeddings @ classifier_weights[:, class_index] + classifier_biases[class_index]
        keep = logits >= score_to_logit(FAST_MIN_SCORE)
        if not np.any(keep):
            continue
        kept_ids = np.asarray(window_ids, dtype=np.int64)[keep]
        selected = pd.DataFrame(
            {
                "_row": np.arange(len(kept_ids), dtype=np.int64),
                "perch_label": label,
                "logit": logits[keep].astype(np.float32),
            }
        )
        frame = attach_metadata(selected, kept_ids, method="fast_search")
        pair_mask = [
            (deployment, label) in allowed_pairs
            for deployment in frame["deployment"].astype(str)
        ]
        frame = frame.loc[pair_mask]
        if not frame.empty:
            candidates.append(frame)

    if not candidates:
        report = {
            "config_hash": config_hash,
            "n_predictions": 0,
            "message": "Ningún candidato superó el umbral.",
        }
        atomic_write_text(base / "run_report.json", json.dumps(report, indent=2))
        return empty_prediction_frame()

    predictions = pd.concat(candidates, ignore_index=True)
    group_columns = ["deployment", partition_column(), "perch_label"]
    predictions = (
        predictions.sort_values("score", ascending=False)
        .groupby(group_columns, dropna=False, sort=False, observed=True)
        .head(FAST_MAX_PER_SPECIES_PER_PARTITION)
        .sort_values(["deployment", partition_column(), "perch_label", "score"],
                     ascending=[True, True, True, False])
        .reset_index(drop=True)
    )

    for (deployment, period), frame in predictions.groupby(
        ["deployment", partition_column()], sort=True, dropna=False, observed=True
    ):
        path = partition_path(base, deployment, period)
        if path.exists() and RESUME and not OVERWRITE_PARTITIONS:
            continue
        atomic_write_parquet(frame, path)

    report = {
        "config_hash": config_hash,
        "n_predictions": int(len(predictions)),
        "n_windows": int(predictions["window_id"].nunique()),
        "n_species": int(predictions["perch_label"].nunique()),
    }
    atomic_write_text(base / "run_report.json", json.dumps(report, indent=2))
    return predictions


if RUN_FAST_SEARCH:
    fast_predictions_df = run_fast_search()
    print(f"Predicciones rápidas conservadas: {len(fast_predictions_df):,}")
    display(fast_predictions_df.head(20))
else:
    fast_predictions_df = empty_prediction_frame()
    print("Búsqueda rápida desactivada.")

## 8. Enfoque 2 — inferencia exhaustiva desde embeddings

Cada partición se procesa por lotes y se escribe inmediatamente. No se acumulan
los logits de toda la base. `MAX_PREDICTIONS_PER_WINDOW` limita el número de
clases guardadas después del umbral; use `None` para conservar todas las clases
que lo superen.

Con `RESUME=True`, un archivo Parquet existente se salta. La escritura usa un
temporal y un rename atómico, por lo que una interrupción no deja una partición
aparentemente completa.

In [ ]:
def batched(values: Sequence[int], size: int) -> Iterable[list[int]]:
    for start in range(0, len(values), size):
        yield list(values[start : start + size])


def run_exhaustive_inference() -> dict[str, int]:
    base = OUTPUT_ROOT / "exhaustive_inference"
    period_column = partition_column()
    groups = windows_df.groupby(
        ["deployment", period_column], sort=True, dropna=False, observed=True
    )
    counters = {"processed": 0, "skipped": 0, "predictions": 0}

    for (deployment, period), partition in tqdm(
        groups, total=groups.ngroups, desc="Particiones exhaustivas"
    ):
        output_path = partition_path(base, deployment, period)
        if output_path.exists() and RESUME and not OVERWRITE_PARTITIONS:
            counters["skipped"] += 1
            continue

        deployment_labels = species_by_deployment_df.loc[
            species_by_deployment_df["deployment"] == deployment,
            "perch_label",
        ].tolist()
        class_indices = np.asarray(
            [head_index[label] for label in deployment_labels], dtype=np.int64
        )
        partition_frames: list[pd.DataFrame] = []
        window_ids = partition["window_id"].astype(np.int64).tolist()

        for batch_ids in batched(window_ids, EMBEDDING_BATCH_SIZE):
            embeddings = np.asarray(db.get_embeddings_batch(batch_ids), dtype=np.float32)
            logits = (
                embeddings @ classifier_weights[:, class_indices]
                + classifier_biases[class_indices]
            )
            selected = select_from_logits(
                logits,
                deployment_labels,
                min_score=EXHAUSTIVE_MIN_SCORE,
                top_k=MAX_PREDICTIONS_PER_WINDOW,
            )
            frame = attach_metadata(
                selected, batch_ids, method="exhaustive_inference"
            )
            if not frame.empty:
                partition_frames.append(frame)

        output = (
            pd.concat(partition_frames, ignore_index=True)
            if partition_frames
            else empty_prediction_frame()
        )
        if not output.empty:
            output = output.sort_values(
                ["window_id", "score"], ascending=[True, False]
            ).reset_index(drop=True)
        atomic_write_parquet(output, output_path)
        counters["processed"] += 1
        counters["predictions"] += len(output)

        partition_report = {
            "config_hash": config_hash,
            "deployment": str(deployment),
            period_column: str(period),
            "n_input_windows": len(window_ids),
            "n_predictions": int(len(output)),
            "n_species": int(output["perch_label"].nunique()) if not output.empty else 0,
        }
        atomic_write_text(
            output_path.with_name("partition_report.json"),
            json.dumps(partition_report, indent=2, ensure_ascii=False),
        )

    atomic_write_text(
        base / "run_report.json",
        json.dumps({"config_hash": config_hash, **counters}, indent=2),
    )
    return counters


if RUN_EXHAUSTIVE_INFERENCE:
    exhaustive_counters = run_exhaustive_inference()
    print(exhaustive_counters)
else:
    exhaustive_counters = {"processed": 0, "skipped": 0, "predictions": 0}
    print("Inferencia exhaustiva desactivada.")

## 9. Resúmenes diarios y horarios

Los resúmenes se construyen leyendo una partición a la vez. `n_detections` es
el número de pares ventana–especie conservados; `n_windows` y `n_recordings`
ayudan a distinguir muchas ventanas de una misma grabación.

In [ ]:
def summarize_method(method_name: str) -> dict[str, Path]:
    base = OUTPUT_ROOT / method_name
    files = sorted(base.glob("deployment=*/**/predictions.parquet"))
    if not files:
        print(f"Sin particiones para {method_name}.")
        return {}

    daily_parts: list[pd.DataFrame] = []
    hourly_parts: list[pd.DataFrame] = []
    for path in tqdm(files, desc=f"Resúmenes {method_name}"):
        frame = pd.read_parquet(
            path,
            columns=[
                "deployment", "date", "hour", "perch_label", "scientific_name",
                "window_id", "recording_id", "score",
            ],
        )
        if frame.empty:
            continue
        for columns, destination in (
            (["deployment", "date", "perch_label", "scientific_name"], daily_parts),
            (["deployment", "hour", "perch_label", "scientific_name"], hourly_parts),
        ):
            summary = (
                frame.dropna(subset=[columns[1]])
                .groupby(columns, dropna=False, observed=True)
                .agg(
                    n_detections=("window_id", "size"),
                    n_windows=("window_id", "nunique"),
                    n_recordings=("recording_id", "nunique"),
                    max_score=("score", "max"),
                    mean_score=("score", "mean"),
                )
                .reset_index()
            )
            destination.append(summary)

    outputs: dict[str, Path] = {}
    for period, parts in (("daily", daily_parts), ("hourly", hourly_parts)):
        if not parts:
            continue
        combined = pd.concat(parts, ignore_index=True)
        period_column = "date" if period == "daily" else "hour"
        # Cada deployment-periodo vive en una sola partición; este groupby también
        # hace el resultado robusto ante futuros esquemas de particionado.
        combined = (
            combined.groupby(
                ["deployment", period_column, "perch_label", "scientific_name"],
                dropna=False,
                observed=True,
            )
            .agg(
                n_detections=("n_detections", "sum"),
                n_windows=("n_windows", "sum"),
                n_recordings=("n_recordings", "sum"),
                max_score=("max_score", "max"),
                mean_score=("mean_score", "mean"),
            )
            .reset_index()
            .sort_values(["deployment", period_column, "max_score"],
                         ascending=[True, True, False])
        )
        path = OUTPUT_ROOT / "summaries" / f"{method_name}_{period}.csv"
        combined.to_csv(path, index=False)
        outputs[period] = path
    return outputs


summary_outputs = {}
if RUN_FAST_SEARCH:
    summary_outputs["fast_search"] = summarize_method("fast_search")
if RUN_EXHAUSTIVE_INFERENCE:
    summary_outputs["exhaustive_inference"] = summarize_method(
        "exhaustive_inference"
    )
summary_outputs

## 10. Cierre y lectura recomendada

- Use **búsqueda rápida** para inspección, descubrimiento y revisión manual de
  candidatos con poco almacenamiento.
- Use **inferencia exhaustiva** para un resultado comparable con la salida
  multiespecie original de Perch y para comparar actividad entre días.
- Ajuste el threshold con un conjunto validado localmente. Un `score` alto no
  reemplaza la verificación acústica y los thresholds no están calibrados de la
  misma forma para todas las especies.
- Revise siempre `species_unmatched.csv`: evita que diferencias taxonómicas o
  nombres obsoletos reduzcan silenciosamente la lista geográfica.
- Para repetir con otra lista o umbral, cambie `RUN_NAME`; así conserva corridas
  anteriores sin mezclar resultados.

In [ ]:
db.close()
print("Base cerrada. Resultados disponibles en:", OUTPUT_ROOT)